In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np
import re

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def find_timing_file(run_folder_path: str, sim_type: str) -> Optional[str]:
    """Finds the ...trace_matched_timing.csv file for a given run."""
    sim_output_dir = os.path.join(run_folder_path, sim_type.lower())
    if not os.path.isdir(sim_output_dir):
        return None
    for f in os.listdir(sim_output_dir):
        if 'trace_matched_timing.csv' in f:
            return os.path.join(sim_output_dir, f)
    return None


In [20]:
import re
import pandas as pd
from plotly.subplots import make_subplots

# --- Configuration ---
base_comparison_folders = [
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/multiple_collectives/'
]
comparison_plot_metric = 'max'  # Can be 'avg' or 'max'

# --- Helper Functions ---
cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}
topo_idx_regex = re.compile(r'(?:ns3|G2)_FoldedClos_16_v(\d+)_Random(?:\.json)?$')

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        h, m, s = map(float, time_str.split(':'))
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0

def process_simulation(folder: str, run_folder: str, sim_type: str, summary_params: dict, workload_name: str, topo_index: int, run_name: str) -> Optional[dict]:
    """Processes a single simulation run, extracts timing data, and returns a result dictionary."""
    timing_file = find_timing_file(run_folder, sim_type)
    if not timing_file:
        return None

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'ns3' and 'node_name' in df.columns:
            df = df[df['node_name'] != 'dummy_node'].copy()

        time_col = 'callback_tick'
        if time_col not in df.columns:
            return None

        df = df[df[time_col] > 100].copy()
        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            return None

        return {
            'workload': workload_name,
            'npu_count': summary_params.get('npus count', 'N/A'),
            'topo_index': topo_index,
            'sim_type': sim_type.upper(),
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'min_time': elapsed_times.min(),
            'std_dev': elapsed_times.std(),
            'execution_time': parse_runtime(summary_params.get('total runtime', '0:0:0.0')),
            'folder': folder,
            'run_folder': run_folder
        }
    except Exception as e:
        print(f"Error processing {sim_type} in {folder}: {e}")
        return None

# --- Data Collection Logic ---
all_run_folders = []
for base_folder in base_comparison_folders:
    for workload_folder in os.listdir(base_folder):
        workload_path = os.path.join(base_folder, workload_folder)
        if os.path.isdir(workload_path):
            all_run_folders.extend([(workload_path, os.path.join(workload_path, d)) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))])

comparison_results = []
for folder, run_folder in sorted(all_run_folders):
    run_summary_path = os.path.join(run_folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    summary_params = parse_config(run_summary_path)
    workload_name = os.path.basename(os.path.dirname(run_folder))
    
    # Process all potential simulation types in the folder
    sim_dirs = [d for d in os.listdir(run_folder) if os.path.isdir(os.path.join(run_folder, d)) and d in ['analytical_unaware', 'g2', 'ns3']]

    for sim_type in sim_dirs:
        topo_index = -1
        run_name = f"{sim_type.replace('_', ' ').title()}"

        if sim_type != 'analytical_unaware':
            topo_key = 'g2 topology file override' if sim_type == 'g2' else 'ns3 topology file override'
            topo_file = summary_params.get(topo_key, '')
            if 'all_paths' in topo_file: continue
            
            match = topo_idx_regex.search(topo_file)
            if not match: continue
            topo_index = int(match.group(1))

            if sim_type == 'ns3':
                ns3_config_file = find_config_file(run_folder)
                if ns3_config_file:
                    ns3_params = parse_config(ns3_config_file)
                    run_name = (
                        f"NS3 (cc:{cc_modes.get(int(ns3_params.get('cc_mode', -1)), 'N/A')}, "
                        f"win:{ns3_params.get('has_win', 'N/A')}, adapt:{ns3_params.get('var_win', 'N/A')}, "
                        f"buf:{ns3_params.get('buffer_size', 'N/A')}, size:{ns3_params.get('packet_payload_size', 'N/A')}), "
                        f"kmax:{ns3_params.get('kmax_map', 'N/A').split(' ')[2]}, kmin:{ns3_params.get('kmin_map', 'N/A').split(' ')[2]}, "
                        f"tinc:{ns3_params.get('rp_timer', 'N/A')}"
                    )
            else: # G2
                run_name = "G2"

        result = process_simulation(folder, run_folder, sim_type, summary_params, workload_name, topo_index, run_name)
        if result:
            comparison_results.append(result)

# --- Plotting Logic ---


In [26]:
comp_df

,workload,npu_count,topo_index,sim_type,run_name,avg_time,max_time,min_time,std_dev,execution_time,folder,run_folder
0,all_gather_size_1073741824_group_2,16,1,G2,G2,7.786778e+07,88991742,44495881,1.989915e+07,0.967082,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
1,all_gather_size_1073741824_group_2,16,2,G2,G2,8.064877e+07,88991742,44495881,1.793686e+07,0.922811,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
2,all_gather_size_1073741824_group_3,16,1,G2,G2,7.508679e+07,88991742,44495881,2.130077e+07,0.836616,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
3,all_gather_size_1073741824_group_3,16,2,G2,G2,8.760125e+07,133487604,44495881,2.868603e+07,0.918286,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
4,all_gather_size_1073741824_group_4,16,1,G2,G2,1.334876e+08,133487623,133487623,0.000000e+00,0.978814,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
...,...,...,...,...,...,...,...,...,...,...,...,...
194,reduce_scatter_size_1073741824_group_9,16,2,NS3,"NS3 (cc:DCQCN, win:0, adapt:0, buf:8, size:100...",1.028779e+08,121746658,90300194,1.169568e+07,2153.625883,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
195,reduce_scatter_size_1073741824_group_9,16,2,NS3,"NS3 (cc:DCQCN, win:0, adapt:0, buf:8, size:100...",9.452521e+07,111880014,84297608,8.325063e+06,2083.832993,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
196,reduce_scatter_size_1073741824_group_9,16,2,NS3,"NS3 (cc:DCQCN, win:0, adapt:0, buf:8, size:100...",9.497475e+07,107158447,82728609,8.065946e+06,2062.563041,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
197,reduce_scatter_size_33554432_group_7,16,1,NS3,"NS3 (cc:DCQCN, win:0, adapt:0, buf:8, size:100...",3.618757e+06,4722053,2315071,6.629693e+05,68.147008,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...


/tmp/ipykernel_917625/282345839.py:2: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



In [29]:
# --- Analysis and Plotting ---
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    ns3_win0 = comp_df[(comp_df['sim_type'] == 'NS3') &
                   comp_df['run_name'].str.contains(r'win:0(,|\b)', regex=True)]
    filtered_comp_df = pd.concat([comp_df[comp_df['sim_type'] == 'G2'], ns3_win0], ignore_index=True)

    # replace comp_df for the analysis (or pass filtered_comp_df into your function)
    _old_comp_df = comp_df
    comp_df = filtered_comp_df
    comp_df = comp_df[(~comp_df['workload'].str.contains('group_6'))&(~comp_df['workload'].str.contains('group_8')&(~comp_df['workload'].str.contains('group_0'))&(~comp_df['workload'].str.contains('group_1')))
                      & (comp_df['topo_index']!=3)]

    def analyze_and_plot_divergence(metric_col, metric_name):
        print(f"--- Analyzing Divergence for: {metric_name} ---")
        
        divergence_data = []
        for (wl, ti), group in comp_df.groupby(['workload', 'topo_index']):
            g2_runs = group[group['sim_type'] == 'G2']
            ns3_runs = group[group['sim_type'] == 'NS3']

            if g2_runs.empty or ns3_runs.empty:
                continue

            g2_time = g2_runs.iloc[0][metric_col]
            best_ns3_run = ns3_runs.loc[ns3_runs[metric_col].idxmin()]
            best_ns3_time = best_ns3_run[metric_col]

            if g2_time < 100 or best_ns3_time < 100:
                continue

            divergence = abs((g2_time - best_ns3_time)/g2_time)
            # Extract info from workload name
            match = re.match(r'([a-zA-Z_]+)_size_(\d+)_group_(\d+)', wl)
            comm_type, comm_size, group_id = "N/A", "N/A", "N/A"
            if match:
                comm_type = match.group(1).replace('_', ' ').title()
                comm_size = int(match.group(2))
                group_id = int(match.group(3))

            divergence_data.append({
                'Workload': wl,
                'Topo_index': ti,
                'Divergence': divergence,
                'G2 Time': g2_time,
                'Best NS3 Time': best_ns3_time,
                'Best NS3 Run': best_ns3_run['run_name'],
                'comm_type': comm_type,
                'comm_size': comm_size,
                'group_id': group_id,
            })

        if not divergence_data:
            print(f"No divergent workloads found for {metric_name}.\n")
            return

        div_df = pd.DataFrame(divergence_data).sort_values(by='Divergence', ascending=False)
        top_10_workloads = div_df.head(40)

        print(f"\n--- Top 10 Most Divergent Workloads ({metric_name}) ---")
        with pd.option_context('display.float_format', '{:,.2f}'.format):
            display(top_10_workloads[['Workload', 'comm_type', 'comm_size', 'group_id', 'G2 Time', 'Best NS3 Time', 'Divergence', 'Topo_index']])

        print(f"\n--- Generating Plots for Top 10 Divergent Workloads ({metric_name}) ---")
        for _, row in top_10_workloads.iterrows():
            wl = row['Workload']
            topo_index = row['Topo_index']
            
            workload_df = comp_df[(comp_df['workload'] == wl) & (comp_df['topo_index'] == topo_index)]
        
            group_df = workload_df

            g2_run = group_df[group_df['sim_type'] == 'G2'].sort_values(by=metric_col).drop_duplicates(subset=['run_name'], keep='first')
            ns3_runs_for_plot = group_df[group_df['sim_type'] == 'NS3'].sort_values(by=metric_col).drop_duplicates(subset=['run_name'], keep='first')
            best_ns3_run_for_plot = ns3_runs_for_plot.iloc[0]

            fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Performance ({metric_name})", "Per-NPU Time Correlation"))

            # Bar plot
            fig.add_trace(go.Bar(x=ns3_runs_for_plot['run_name'], y=ns3_runs_for_plot[metric_col], name='NS3 Runs'), row=1, col=1)
            fig.add_hline(y=
            g2_run[metric_col].values[0], line_dash="dot", annotation_text=f"G2 Time", row=1, col=1)

            # Scatter plot
            g2_timing_file = find_timing_file(g2_run['run_folder'].values[0], 'G2')
            ns3_timing_file = find_timing_file(best_ns3_run_for_plot['run_folder'], 'NS3')
            if g2_timing_file and ns3_timing_file:
                try:
                    df_g2 = pd.read_csv(g2_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'g2_time'})
                    df_ns3 = pd.read_csv(ns3_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'ns3_time'})
                    merged_df = pd.merge(df_ns3, df_g2, on='sys_id')
                    merged_df = merged_df[(merged_df['ns3_time'] >= 100) & (merged_df['g2_time'] >= 100)]
                    
                    fig.add_trace(go.Scatter(x=merged_df['ns3_time'], y=merged_df['g2_time'], mode='markers', name='NPU Times'), row=1, col=2)
                    min_val = min(merged_df['ns3_time'].min(), merged_df['g2_time'].min())
                    max_val = max(merged_df['ns3_time'].max(), merged_df['g2_time'].max())
                    fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', name='y=x'), row=1, col=2)
                except Exception as e:
                    print(f"Could not create scatter plot for {wl}: {e}")
            
            plot_title = (f'<b>{wl}</b><br>'
                          f'Metric: {metric_name}<br>'
                          f'Comm: {row["comm_type"]}, Size: {row["comm_size"]}, Group: {row["group_id"]}')

            fig.update_layout(title_text=plot_title, height=700, margin=dict(t=140))
            fig.show()
        print("\n" + "="*80 + "\n")
        return div_df


    # --- Run analysis for each metric ---
    div_df_avg = analyze_and_plot_divergence('avg_time', 'Average Time')
    # div_df_max = analyze_and_plot_divergence('max_time', 'Maximum Time')
    # div_df_min = analyze_and_plot_divergence('min_time', 'Minimum Time')

else:
    print("No comparison results to process.")


--- Analyzing Divergence for: Average Time ---

--- Top 10 Most Divergent Workloads (Average Time) ---


/tmp/ipykernel_917625/1787655732.py:5: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



,Workload,comm_type,comm_size,group_id,G2 Time,Best NS3 Time,Divergence,Topo_index
16,all_reduce_size_33554432_group_9,All Reduce,33554432,9,"5,259,701.38","8,780,713.62",0.67,2
38,reduce_scatter_size_33554432_group_7,Reduce Scatter,33554432,7,"2,516,910.88","4,049,742.69",0.61,2
15,all_reduce_size_33554432_group_9,All Reduce,33554432,9,"5,540,013.75","8,643,690.56",0.56,1
32,all_to_all_size_33554432_group_9,All To All,33554432,9,"10,657,091.50","16,603,209.19",0.56,2
31,all_to_all_size_33554432_group_9,All To All,33554432,9,"10,949,798.38","16,088,969.00",0.47,1
37,reduce_scatter_size_33554432_group_7,Reduce Scatter,33554432,7,"2,463,463.62","3,618,756.62",0.47,1
14,all_reduce_size_33554432_group_2,All Reduce,33554432,2,"2,770,756.81","3,618,355.62",0.31,2
13,all_reduce_size_33554432_group_2,All Reduce,33554432,2,"2,555,995.69","3,330,694.12",0.30,1
3,all_gather_size_33554432_group_4,All Gather,33554432,4,"7,647,758.00","9,818,156.00",0.28,2
30,all_to_all_size_33554432_group_4,All To All,33554432,4,"4,171,528.00","5,303,754.62",0.27,2



--- Generating Plots for Top 10 Divergent Workloads (Average Time) ---


In [4]:
pd.DataFrame(comparison_results)

,workload,npu_count,topo_index,sim_type,run_name,avg_time,max_time,min_time,std_dev,execution_time,folder,run_folder
0,all_gather_size_1073741824_group_0,16,-1,ANALYTICAL_UNAWARE,Analytical Unaware,6.000002e+08,600000160,600000160,0.000000e+00,1.219010,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
1,all_gather_size_1073741824_group_0,16,1,G2,G2,6.674381e+08,667438075,667438075,0.000000e+00,0.843086,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
2,all_gather_size_1073741824_group_0,16,2,G2,G2,6.674381e+08,667438075,667438075,0.000000e+00,0.851751,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
3,all_gather_size_1073741824_group_2,16,-1,ANALYTICAL_UNAWARE,Analytical Unaware,4.000002e+07,40000020,40000020,0.000000e+00,1.288481,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
4,all_gather_size_1073741824_group_2,16,1,G2,G2,7.786778e+07,88991742,44495881,1.989915e+07,0.967082,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
...,...,...,...,...,...,...,...,...,...,...,...,...
466,reduce_scatter_size_33554432_group_7,16,2,NS3,"NS3 (cc:DCQCN, win:1, adapt:1, buf:8, size:100...",3.149798e+06,3989951,1569986,6.417542e+05,62.274038,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
467,reduce_scatter_size_33554432_group_7,16,2,NS3,"NS3 (cc:DCQCN, win:1, adapt:1, buf:8, size:100...",3.149798e+06,3989951,1569986,6.417542e+05,61.414713,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
468,reduce_scatter_size_33554432_group_9,16,-1,ANALYTICAL_UNAWARE,Analytical Unaware,1.120122e+06,1120122,1120122,0.000000e+00,1.304370,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
469,reduce_scatter_size_33554432_group_9,16,1,G2,G2,2.495389e+06,3087072,1586909,5.214891e+05,1.037235,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
